# Task 60: NLP and text classification

Reference: https://medium.com/towards-artificial-intelligence/natural-language-processing-nlp-with-python-tutorial-for-beginners-1f54e610a1a0

A small explicitly labeled toy sentiment example, with held-out test sentences.

In [1]:
# Task 60: NLP and text classification
# Reference: https://medium.com/towards-artificial-intelligence/natural-language-processing-nlp-with-python-tutorial-for-beginners-1f54e610a1a0

from pathlib import Path
import nltk

# The download includes these resources, so this setup works offline.
# Running an individual file without the data folder requires internet once.
base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
resource_folder = base / 'data' / 'nltk_data'
nltk.data.path.insert(0, str(resource_folder))
resources = {
    'corpora/stopwords': 'stopwords',
    'tokenizers/punkt_tab/english/': 'punkt_tab',
    'taggers/averaged_perceptron_tagger_eng/': 'averaged_perceptron_tagger_eng',
    'corpora/wordnet.zip': 'wordnet',
}
for resource_path, package in resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        if not nltk.download(package, download_dir=str(resource_folder), quiet=True):
            raise RuntimeError('Could not download NLTK resource: ' + package)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

text = 'Natural language processing helps computers understand human languages.'
tokens = word_tokenize(text.lower())
stop_words = set(stopwords.words('english'))
filtered = [token for token in tokens if token.isalpha() and token not in stop_words]
stemmer = PorterStemmer()
print('Tokens:', tokens)
print('Without punctuation and stop words:', filtered)
print('Stems:', [stemmer.stem(word) for word in filtered])

# Tiny labeled toy dataset: these results do not represent real-world accuracy.
train_texts = [
    'excellent movie with a wonderful story', 'great acting and brilliant scenes',
    'I loved this amazing film', 'a good enjoyable movie',
    'fantastic performance and excellent direction', 'wonderful cast and great music',
    'terrible movie with a boring story', 'bad acting and awful scenes',
    'I hated this disappointing film', 'a poor boring movie',
    'awful performance and terrible direction', 'bad cast and horrible music'
]
train_labels = [1] * 6 + [0] * 6
# Separate held-out sentences, never passed to fit().
test_texts = ['a wonderful enjoyable film', 'bad boring acting',
              'excellent music and great scenes', 'a terrible awful film']
test_labels = [1, 0, 1, 0]
model = make_pipeline(TfidfVectorizer(stop_words='english'),
                      LogisticRegression(C=10, random_state=13))
model.fit(train_texts, train_labels)
predictions = model.predict(test_texts)
print('Training/test documents:', len(train_texts), len(test_texts))
print('Vocabulary size:', len(model.named_steps['tfidfvectorizer'].vocabulary_))
for sentence, label in zip(test_texts, predictions):
    print(sentence, '->', 'positive' if label == 1 else 'negative')
print('Toy holdout accuracy:', round(float(accuracy_score(test_labels, predictions)), 4))


Tokens: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'languages', '.']
Without punctuation and stop words: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'languages']
Stems: ['natur', 'languag', 'process', 'help', 'comput', 'understand', 'human', 'languag']
Training/test documents: 12 4
Vocabulary size: 26
a wonderful enjoyable film -> positive
bad boring acting -> negative
excellent music and great scenes -> positive
a terrible awful film -> negative
Toy holdout accuracy: 1.0
